# Coffee Shop Management Portal

**Assignment 3: End-to-End Portal Design**

**Course:** Programming for Business Lab    

**Group members:** Amna Hassan (25L-5126) and Abeera Mudasssar (25L-5040)

**Introduction:** This notebook contains a portal for a small coffee shop. Staff can log in, take customer orders, print bills, search old orders, manage inventory and view sales reports. The manager can also change the menu.

All data is stored in JSON files (`menu.json`, `orders.json`, `inventory.json`), so nothing is lost when the program is closed.

**How to use this notebook:** run the cells from top to bottom. Cells 1 to 12 define the program. The test cells after that show sample runs. The last cell starts the real interactive portal.

## 1. Imports and constants
We use `json` for saving data, `os` for working with files, `copy` so default data is never changed by accident, and `datetime` for order dates and times.
`USERS` stores the login details and the role of each user.

In [2]:
import json 
import os 
import copy 
from datetime import datetime

# File names used for permanent storage
MENU_FILE = "menu.json"
ORDERS_FILE = "orders.json"
INVENTORY_FILE = "inventory.json"
REPORT_FILE = "sales_report.txt"

TAX_RATE = 0.10          
LOW_STOCK_LIMIT = 100   

USERS = {
    "manager": {"password": "manager123", "role": "manager"},
    "staff": {"password": "staff123", "role": "employee"},
}

# "uses" = how much inventory one unit of the item uses.
# no uses for the item "Chocolate Muffin and "Club Sandwich" as too many ingredients would be required.
DEFAULT_MENU = {
    "1": {"name": "Espresso", "category": "Coffee", "price": 250.0,
          "uses": {"coffee": 18, "cups": 1}},
    "2": {"name": "Cappuccino", "category": "Coffee", "price": 350.0,
          "uses": {"coffee": 18, "milk": 150, "sugar": 5, "cups": 1}},
    "3": {"name": "Latte", "category": "Coffee", "price": 380.0,
          "uses": {"coffee": 18, "milk": 200, "sugar": 5, "cups": 1}},
    "4": {"name": "Iced Tea", "category": "Beverage", "price": 220.0,
          "uses": {"sugar": 10, "cups": 1}},
    "5": {"name": "Hot Chocolate", "category": "Beverage", "price": 320.0,
          "uses": {"milk": 200, "sugar": 15, "cups": 1}},
    "6": {"name": "Chocolate Muffin", "category": "Food", "price": 200.0,
          "uses": {}},
    "7": {"name": "Club Sandwich", "category": "Food", "price": 450.0,
          "uses": {}},
}

# Quantity and the unit inventory is measured in
DEFAULT_INVENTORY = {
    "coffee": {"quantity": 5000, "unit": "g"},
    "milk": {"quantity": 20000, "unit": "ml"},
    "sugar": {"quantity": 3000, "unit": "g"},
    "cups": {"quantity": 500, "unit": "pcs"},
} 

## 2. File handling functions
`load_data()` reads a JSON file and `save_data()` writes one.

Errors handled here:
- `FileNotFoundError`: the file does not exist yet, so we create it with default data.
- `json.JSONDecodeError`: the file is damaged. We keep a `.bak` backup of it and start with default data, so the old file is not lost.
- `PermissionError` / `OSError`: the file cannot be read or written. The program prints a message and keeps running.

In [3]:
def save_data(filename, data):
    try:
        with open(filename, "w") as f:
            json.dump(data, f, indent=4)
        return True
    except PermissionError:
        print(f"Error: no permission to write to {filename}.")
        return False
    except OSError as e:
        print(f"Error: could not save {filename} ({e}).")
        return False

def load_data(filename, default):
    try:
        with open(filename, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"{filename} not found. Creating a new file.")
        data = copy.deepcopy(default)
        save_data(filename, data)
        return data
    except json.JSONDecodeError:
        print(f"Warning: {filename} is corrupted. Backup saved as {filename}.bak")
        try:
            os.replace(filename, filename + ".bak")
        except OSError:
            pass
        data = copy.deepcopy(default)
        save_data(filename, data)
        return data
    except PermissionError:
        print(f"Error: no permission to read {filename}. Using default data.")
        return copy.deepcopy(default)

## 3. Input helper functions
These functions keep asking until the user types a valid number. A `ValueError` is raised when text like `"abc"` is converted with `int()` or `float()`, and we catch it with `try-except` so the program does not crash.

In [4]:
def enter_int(message, minimum=None):
    while True:
        try:
            value = int(input(message))
            if minimum is not None and value < minimum:
                print(f"Please enter a number that is {minimum} or more.")
            else:
                return value
        except ValueError:
            print("Invalid input. Please enter a whole number.")

def enter_float(message):
    while True:
        try:
            value = float(input(message))
            if value <= 0:
                print("Price must be greater than 0.")
            else:
                return value
        except ValueError:
            print("Invalid input. Please enter a number (e.g. 250.50).")

def line(character ="-", width=60):
    print(character * width)

## 4. Module 1: User login
`user_login()` gives the user 3 attempts. It returns `(username, role)` if the login works, or `None` if all attempts fail. The role decides what the user is allowed to do (only the manager can update the menu).

In [5]:
def user_login():
    line("=")
    print("        COFFEE SHOP MANAGEMENT PORTAL")
    line("=")
    attempts = 3
    while attempts > 0:
        username = input("Username: ").strip().lower()
        password = input("Password: ").strip()

        if username in USERS and USERS[username]["password"] == password:
            role = USERS[username]["role"]
            print(f"\nWelcome, {username}! (Role: {role})")
            return username, role

        attempts = attempts - 1
        print(f"Incorrect username or password. Attempts left: {attempts}\n")

    print("Too many failed attempts.")
    return None

## 5. Module 2: View menu
Items are grouped by category (Coffee, Beverage, Food). A `for` loop first collects the categories, then a second loop prints the items in each one.

In [6]:
def display_menu(menu):
    print()
    line("=")
    print("                    MENU")
    line("=")

    categories = []
    for item in menu.values():
        if item["category"] not in categories:
            categories.append(item["category"])

    if len(categories) == 0:
        print("The menu is empty.")
        return

    for category in categories:
        print(f"\n--- {category.upper()} ---")
        for item_id in menu:
            item = menu[item_id]
            if item["category"] == category:
                print(f"  {item_id:>3}. {item['name']:<22} Rs. {item['price']:>8.2f}")
    line("=")

## 6. Inventory helpers and billing
- `find_missing_supply()` checks if there is enough stock to make an item. It returns the name of the missing supply, or `None` if everything is available.
- `decrease_stock()` reduces the inventory after an order.
- `calculate_bill()` returns the subtotal, tax and total.
- `print_receipt()` prints a formatted bill.

In [7]:
def find_missing_supply(item, qty, inventory):
    for supply in item["uses"]:
        needed = item["uses"][supply] * qty
        if supply not in inventory:
            return supply
        if inventory[supply]["quantity"] < needed:
            return supply
    return None

def decrease_stock(item, qty, inventory):
    for supply in item["uses"]:
        if supply in inventory:
            inventory[supply]["quantity"] -= item["uses"][supply] * qty

def calculate_bill(order_items):
    subtotal = 0.0
    for entry in order_items:
        subtotal = subtotal + entry["price"] * entry["quantity"]
    tax = subtotal * TAX_RATE
    total = subtotal + tax
    return round(subtotal, 2), round(tax, 2), round(total, 2)

def print_receipt(order):
    print()
    line("=")
    print("                   RECEIPT")
    line("=")
    print(f"Order ID : {order['order_id']}")
    print(f"Customer : {order['customer']}")
    print(f"Date     : {order['date']}  {order['time']}")
    print(f"Served by: {order['cashier']}")
    line()
    print(f"{'Item':<22}{'Qty':>5}{'Price':>10}{'Total':>12}")
    line()
    for entry in order["items"]:
        row_total = entry["price"] * entry["quantity"]
        print(f"{entry['name']:<22}{entry['quantity']:>5}"
              f"{entry['price']:>10.2f}{row_total:>12.2f}")
    line()
    tax_label = "Tax (" + str(int(TAX_RATE * 100)) + "%)"
    print(f"{'Subtotal':<37}{order['subtotal']:>13.2f}")
    print(f"{tax_label:<37}{order['tax']:>13.2f}")
    print(f"{'TOTAL':<37}{order['total']:>13.2f}")
    line("=")


def show_bill(orders):
    order_id = enter_int("Enter Order ID: ", minimum=1)
    for order in orders:
        if order["order_id"] == order_id:
            print_receipt(order)
            return
    print("Order not found.")

## 7. Module 3: Place an order (`add_record`)
This module is split in two parts so it is easy to test:
- `create_order()` builds the order record, reduces the stock and adds it to the orders list (no input or printing).
- `add_record()` is the interactive part. It asks the cashier for items, checks stock, then calls `create_order()`, saves the files and prints the receipt.

In [8]:
def check_low_stock(inventory):
    for name in inventory:
        info = inventory[name]
        if info["quantity"] < LOW_STOCK_LIMIT:
            print(f"** LOW STOCK WARNING: {name} ({info['quantity']} {info['unit']} left) **")


def create_order(customer, order_items, menu, orders, inventory, cashier):
    for entry in order_items:
        decrease_stock(menu[entry["item_id"]], entry["quantity"], inventory)

    subtotal, tax, total = calculate_bill(order_items)
    now = datetime.now()

    # next order id = highest existing id + 1
    new_id = 1
    for old_order in orders:
        if old_order["order_id"] >= new_id:
            new_id = old_order["order_id"] + 1

    order = {
        "order_id": new_id,
        "customer": customer,
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "cashier": cashier,
        "items": order_items,
        "subtotal": subtotal,
        "tax": tax,
        "total": total,
    }
    orders.append(order)
    return order


def add_record(menu, orders, inventory, cashier):
    display_menu(menu)
    customer = input("Customer name (press Enter for 'Walk-in'): ").strip()
    if customer == "":
        customer = "Walk-in"

    order_items = []

    while True:
        item_id = input("\nEnter item number (or 'done' to finish): ").strip().lower()

        if item_id == "done":
            break
        if item_id not in menu:
            print("Invalid item number. Please choose from the menu.")
            continue

        item = menu[item_id]
        qty = enter_int(f"Quantity of {item['name']}: ", minimum=1)

        # how many of this item are already in the current order?
        already = 0
        for entry in order_items:
            if entry["item_id"] == item_id:
                already = already + entry["quantity"]

        missing = find_missing_supply(item, already + qty, inventory)
        if missing is not None:
            print(f"Sorry, not enough '{missing}' in stock for {item['name']}.")
            continue

        # if the item is already in the order, just increase its quantity
        found = False
        for entry in order_items:
            if entry["item_id"] == item_id:
                entry["quantity"] = entry["quantity"] + qty
                found = True
        if not found:
            order_items.append({"item_id": item_id, "name": item["name"],
                                "price": item["price"], "quantity": qty})
        print(f"Added {qty} x {item['name']}")

    if len(order_items) == 0:
        print("No items were added. Order cancelled.")
        return None

    order = create_order(customer, order_items, menu, orders, inventory, cashier)
    save_data(ORDERS_FILE, orders)
    save_data(INVENTORY_FILE, inventory)

    print("\nOrder placed successfully!")
    print_receipt(order)
    check_low_stock(inventory)
    return order

## 8. Module 4: Search orders (`search_record`)
`find_orders()` does the actual searching with a `for` loop and returns the matching orders. `search_record()` asks the user how they want to search and prints the results.

In [9]:
def find_orders(orders, order_id=None, name=None):
    found = []
    for order in orders:
        if order_id is not None and order["order_id"] == order_id:
            found.append(order)
        elif name is not None and name.lower() in order["customer"].lower():
            found.append(order)
    return found

def search_record(orders):
    if len(orders) == 0:
        print("No orders have been placed yet.")
        return

    print("\n1. Search by Order ID")
    print("2. Search by Customer Name")
    choice = input("Choose: ").strip()

    if choice == "1":
        order_id = get_int("Enter Order ID: ", minimum=1)
        found = find_orders(orders, order_id=order_id)
    elif choice == "2":
        name = input("Enter customer name: ").strip()
        found = find_orders(orders, name=name)
    else:
        print("Invalid choice.")
        return

    if len(found) == 0:
        print("No matching orders found.")
    else:
        print(f"\n{len(found)} order(s) found:")
        for order in found:
            print_receipt(order)

## 9. Module 5: Inventory management
The user can view the stock levels (supplies below the limit are marked `LOW`) and restock existing supplies or add a new one.

In [10]:
def display_inventory(inventory):
    print()
    line("=")
    print("                 INVENTORY")
    line("=")
    print(f"{'Supply':<15}{'Quantity':>12}  {'Unit':<6}{'Status'}")
    line()
    for name in inventory:
        info = inventory[name]
        if info["quantity"] < LOW_STOCK_LIMIT:
            status = "LOW"
        else:
            status = "OK"
        print(f"{name:<15}{info['quantity']:>12}  {info['unit']:<6}{status}")
    line("=")

def restock_inventory(inventory):
    display_inventory(inventory)
    name = input("Supply name to restock (or a new supply name): ").strip().lower()
    if name == "":
        print("Supply name cannot be empty.")
        return

    if name in inventory:
        amount = enter_int(f"Amount to add ({inventory[name]['unit']}): ", minimum=1)
        inventory[name]["quantity"] += amount
    else:
        unit = input("New supply. Enter its unit (g, ml, pcs): ").strip()
        amount = enter_int("Starting quantity: ", minimum=1)
        inventory[name] = {"quantity": amount, "unit": unit}

    save_data(INVENTORY_FILE, inventory)
    print(f"Inventory updated. {name} now has {inventory[name]['quantity']} "
          f"{inventory[name]['unit']}.")

def inventory_management(inventory):
    while True:
        print("\n--- INVENTORY MANAGEMENT ---")
        print("1. View inventory")
        print("2. Restock / add supply")
        print("3. Back")
        choice = input("Choose: ").strip()

        if choice == "1":
            display_inventory(inventory)
        elif choice == "2":
            restock_inventory(inventory)
        elif choice == "3":
            break
        else:
            print("Invalid choice.")

## 10. Module 6: Update menu (manager only)
The role returned by `user_login()` is checked first. Employees get "Access denied". The manager can add items (and say which supplies they use, so stock is deducted correctly), change prices, or remove items.

In [11]:
def update_menu(menu, inventory, role):
    if role != "manager":
        print("Access denied. Only the manager can update the menu.")
        return

    while True:
        print("\n--- UPDATE MENU ---")
        print("1. Add new item")
        print("2. Change item price")
        print("3. Remove item")
        print("4. Back")
        choice = input("Choose: ").strip()

        if choice == "1":
            name = input("Item name: ").strip()
            if name == "":
                print("Name cannot be empty.")
                continue
            category = input("Category (Coffee / Beverage / Food): ").strip().title()
            if category == "":
                category = "Food"
            price = enter_float("Price: ")

            # ask how much of each supply one unit of this item uses
            uses = {}
            for supply in inventory:
                amount = enter_int(f"{supply} used per item ({inventory[supply]['unit']}, 0 if none): ",
                                 minimum=0)
                if amount > 0:
                    uses[supply] = amount

            # new id = highest existing id + 1
            highest = 0
            for key in menu:
                if int(key) > highest:
                    highest = int(key)
            new_id = str(highest + 1)

            menu[new_id] = {"name": name, "category": category,
                            "price": price, "uses": uses}
            save_data(MENU_FILE, menu)
            print(f"'{name}' added with item number {new_id}.")

        elif choice == "2":
            display_menu(menu)
            item_id = input("Item number to change: ").strip()
            if item_id in menu:
                new_price = enter_float(f"New price for {menu[item_id]['name']}: ")
                menu[item_id]["price"] = new_price
                save_data(MENU_FILE, menu)
                print("Price updated.")
            else:
                print("Item not found.")

        elif choice == "3":
            display_menu(menu)
            item_id = input("Item number to remove: ").strip()
            if item_id in menu:
                removed = menu.pop(item_id)
                save_data(MENU_FILE, menu)
                print(f"'{removed['name']}' removed from the menu.")
            else:
                print("Item not found.")

        elif choice == "4":
            break
        else:
            print("Invalid choice.")

## 11. Module 7: Sales report (`generate_report`)
`calculate_totals()` loops over all orders and collects the numbers (orders and revenue per day, quantity sold per item). `generate_report()` prints them and also saves a copy in `sales_report.txt`.

In [14]:
def calculate_totals(orders):
    revenue_per_day = {}
    orders_per_day = {}
    item_counts = {}
    total_revenue = 0.0

    for order in orders:
        day = order["date"]
        revenue_per_day[day] = revenue_per_day.get(day, 0.0) + order["total"]
        orders_per_day[day] = orders_per_day.get(day, 0) + 1
        total_revenue = total_revenue + order["total"]
        for entry in order["items"]:
            name = entry["name"]
            item_counts[name] = item_counts.get(name, 0) + entry["quantity"]

    return {"total_orders": len(orders), "total_revenue": round(total_revenue, 2),
            "revenue_per_day": revenue_per_day, "orders_per_day": orders_per_day,
            "item_counts": item_counts}


def get_quantity(pair):
    return pair[1]

def generate_report(orders):
    if len(orders) == 0:
        print("No sales data available yet.")
        return

    totals = calculate_totals(orders)
    today = datetime.now().strftime("%Y-%m-%d")

    lines = []
    lines.append("=" * 60)
    lines.append("                SALES REPORT")
    lines.append("=" * 60)
    lines.append(f"Today ({today})")
    lines.append(f"  Orders  : {totals['orders_per_day'].get(today, 0)}")
    lines.append(f"  Revenue : Rs. {totals['revenue_per_day'].get(today, 0.0):.2f}")
    lines.append("-" * 60)
    lines.append("Daily revenue:")
    for day in sorted(totals["revenue_per_day"]):
        lines.append(f"  {day}  |  {totals['orders_per_day'][day]:>3} orders  |  "
                     f"Rs. {totals['revenue_per_day'][day]:>10.2f}")
    lines.append("-" * 60)
    lines.append(f"All time orders  : {totals['total_orders']}")
    lines.append(f"All time revenue : Rs. {totals['total_revenue']:.2f}")
    lines.append("-" * 60)
    lines.append("Items sold (highest first):")
    ranked = sorted(totals["item_counts"].items(), key=get_quantity, reverse=True)
    for name, qty in ranked:
        lines.append(f"  {name:<22}{qty:>5}")
    lines.append("=" * 60)

    for text in lines:
        print(text)

    try:
        with open(REPORT_FILE, "w") as f:
            f.write("\n".join(lines))
        print(f"(Report also saved to {REPORT_FILE})")
    except OSError as e:
        print(f"Could not save the report file ({e}).")

## 12. Main menu and program loop
`main()` loads the data, logs the user in, and then runs a `while True` loop that keeps showing the menu until the user picks option 8.

The `try-except-finally` block makes sure the program closes safely: if the user presses Ctrl+C or the input stream ends (`KeyboardInterrupt`, `EOFError`), we catch it, and the `finally` part always saves all data before the program ends.

In [15]:
def display_main_menu(username, role):
    print()
    line("=")
    print(f"   MAIN MENU   (User: {username} | Role: {role})")
    line("=")
    print("1. View Menu")
    print("2. Place an Order")
    print("3. Reprint Bill (existing order)")
    print("4. Search Orders")
    print("5. Inventory Management")
    print("6. Update Menu (manager only)")
    print("7. Sales Report")
    print("8. Logout / Exit")
    line("=")


def main():
    menu = load_data(MENU_FILE, DEFAULT_MENU)
    orders = load_data(ORDERS_FILE, [])
    inventory = load_data(INVENTORY_FILE, DEFAULT_INVENTORY)

    try:
        session = user_login()
        if session is None:
            return
        username, role = session

        while True:
            display_main_menu(username, role)
            choice = input("Enter your choice (1-8): ").strip()

            if choice == "1":
                display_menu(menu)
            elif choice == "2":
                add_record(menu, orders, inventory, username)
            elif choice == "3":
                show_bill(orders)
            elif choice == "4":
                search_record(orders)
            elif choice == "5":
                inventory_management(inventory)
            elif choice == "6":
                update_menu(menu, inventory, role)
            elif choice == "7":
                generate_report(orders)
            elif choice == "8":
                print("\nLogging out. Goodbye!")
                break
            else:
                print("Invalid choice. Please enter a number from 1 to 8.")
    except (KeyboardInterrupt, EOFError):
        print("\nInput stopped. Closing the portal safely.")
    finally:
        save_data(MENU_FILE, menu)
        save_data(ORDERS_FILE, orders)
        save_data(INVENTORY_FILE, inventory)
        print("All data saved. Session closed.")

# Sample test runs
The cells below test the program without typing anything by hand. Real portal files (`orders.json` etc.) are not touched: the tests use their own `test_*.json` files, which are deleted at the end.

## Test 1: File creation (first run)
On the first run the three data files do not exist, so `load_data()` creates them with the default data.

In [16]:
menu = load_data(MENU_FILE, DEFAULT_MENU)
orders = load_data(ORDERS_FILE, [])
inventory = load_data(INVENTORY_FILE, DEFAULT_INVENTORY)

print("Files in this folder:", [f for f in sorted(os.listdir(".")) if f.endswith(".json")])
print("\ninventory.json contains:")
with open(INVENTORY_FILE) as f:
    print(f.read())

Files in this folder: ['inventory.json', 'menu.json', 'orders.json']

inventory.json contains:
{
    "coffee": {
        "quantity": 5000,
        "unit": "g"
    },
    "milk": {
        "quantity": 19800,
        "unit": "ml"
    },
    "sugar": {
        "quantity": 2985,
        "unit": "g"
    },
    "cups": {
        "quantity": 499,
        "unit": "pcs"
    }
}


## Test 2: File error handling
We test a missing file, a corrupted file, and a path that cannot be written to.

In [17]:
print("--- Missing file ---")
data = load_data("test_missing.json", {"hello": 1})
print("Loaded:", data)

print("\n--- Corrupted file ---")
with open("test_bad.json", "w") as f:
    f.write("{this is not valid json")
data = load_data("test_bad.json", {"hello": 2})
print("Loaded:", data)
print("Backup file exists:", os.path.exists("test_bad.json.bak"))

print("\n--- Cannot write (a folder with the same name as the file) ---")
os.makedirs("test_folder", exist_ok=True)
saved = save_data("test_folder", {"x": 1})
print("Saved successfully?", saved)

--- Missing file ---
test_missing.json not found. Creating a new file.
Loaded: {'hello': 1}

--- Corrupted file ---
Loaded: {'hello': 2}
Backup file exists: True

--- Cannot write (a folder with the same name as the file) ---
Error: no permission to write to test_folder.
Saved successfully? False


## Test 3: Input validation (`ValueError`)
`run_with_inputs()` is a small helper that feeds pre-written answers to `input()`, so we can test interactive functions automatically. Here the "user" types text, a negative number and a zero before finally entering a valid quantity.

In [18]:
import builtins

def run_with_inputs(func, answers, *args):
    answers = list(answers)
    real_input = builtins.input

    def fake_input(prompt=""):
        value = answers.pop(0)
        print(prompt + value)
        return value

    builtins.input = fake_input
    try:
        return func(*args)
    finally:
        builtins.input = real_input

result = run_with_inputs(get_int, ["abc", "-5", "0", "3"], "Quantity: ", 1)
print("enter_int returned:", result)

result = run_with_inputs(get_float, ["free", "-10", "250.5"], "Price: ")
print("enter_float returned:", result)

Quantity: abc
Invalid input. Please enter a whole number.
Quantity: -5
Please enter a number that is 1 or more.
Quantity: 0
Please enter a number that is 1 or more.
Quantity: 3
enter_int returned: 3
Price: free
Invalid input. Please enter a number (e.g. 250.50).
Price: -10
Price must be greater than 0.
Price: 250.5
enter_float returned: 250.5


## Test 4: Login
Wrong username, wrong password, then the correct manager details. A second test uses three wrong attempts. (The staff account would log in as role `employee`.)

In [19]:
print("--- Login succeeds on 3rd try ---")
session = run_with_inputs(user_login, ["abeera", "wrong", "manager", "badpass", "manager", "admin123"])
print("Returned:", session)

print("\n--- Three wrong attempts ---")
session = run_with_inputs(user_login, ["a", "b", "c", "d", "e", "f"])
print("Returned:", session)

--- Login succeeds on 3rd try ---
        COFFEE SHOP MANAGEMENT PORTAL
Username: abeera
Password: wrong
Incorrect username or password. Attempts left: 2

Username: manager
Password: badpass
Incorrect username or password. Attempts left: 1

Username: manager
Password: admin123
Incorrect username or password. Attempts left: 0

Too many failed attempts.
Returned: None

--- Three wrong attempts ---
        COFFEE SHOP MANAGEMENT PORTAL
Username: a
Password: b
Incorrect username or password. Attempts left: 2

Username: c
Password: d
Incorrect username or password. Attempts left: 1

Username: e
Password: f
Incorrect username or password. Attempts left: 0

Too many failed attempts.
Returned: None


## Test 5: Billing and stock checks
Order: 2 x Cappuccino (Rs. 350) and 1 x Chocolate Muffin (Rs. 200).
Expected subtotal = 900.00, tax (10%) = 90.00, total = 990.00.

In [20]:
sample_items = [
    {"item_id": "2", "name": "Cappuccino", "price": 350.0, "quantity": 2},
    {"item_id": "6", "name": "Chocolate Muffin", "price": 200.0, "quantity": 1},
]
print("calculate_bill =", calculate_bill(sample_items))

print("\nStock check, 10 Lattes     =", find_missing_supply(menu["3"], 10, inventory))
print("Stock check, 300 Espressos =", find_missing_supply(menu["1"], 300, inventory),
      "(None means enough stock, otherwise the missing supply is shown)")

calculate_bill = (900.0, 90.0, 990.0)

Stock check, 10 Lattes     = None
Stock check, 300 Espressos = coffee (None means enough stock, otherwise the missing supply is shown)


## Test 6: Placing orders
For these tests we point the program at test files so the real data is not changed. First `create_order()` is called directly. Then `add_record()` is run with simulated typing. The simulated cashier enters a bad item number, a bad quantity (`x`) and finishes with `done`.

In [21]:
# using separate test files for this section
ORDERS_FILE = "test_orders.json"
INVENTORY_FILE = "test_inventory.json"

test_orders = []
test_inventory = copy.deepcopy(DEFAULT_INVENTORY)

# 1) direct call
first = create_order("Sara", sample_items, menu, test_orders, test_inventory, "staff")
print("Created order", first["order_id"], "for", first["customer"], "- total", first["total"])
print("Milk left after order:", test_inventory["milk"]["quantity"], "ml (started at 20000)")

# 2) interactive call with simulated input
answers = ["Ali", "2", "2", "6", "x", "1", "99", "done"]
second = run_with_inputs(add_record, answers, menu, test_orders, test_inventory, "staff")

Created order 1 for Sara - total 990.0
Milk left after order: 19700 ml (started at 20000)

                    MENU

--- COFFEE ---
    1. Espresso               Rs.   250.00
    2. Cappuccino             Rs.   350.00
    3. Latte                  Rs.   380.00

--- BEVERAGE ---
    4. Iced Tea               Rs.   220.00
    5. Hot Chocolate          Rs.   320.00

--- FOOD ---
    6. Chocolate Muffin       Rs.   200.00
    7. Club Sandwich          Rs.   450.00
Customer name (press Enter for 'Walk-in'): Ali

Enter item number (or 'done' to finish): 2
Quantity of Cappuccino: 2
Added 2 x Cappuccino

Enter item number (or 'done' to finish): 6
Quantity of Chocolate Muffin: x
Invalid input. Please enter a whole number.
Quantity of Chocolate Muffin: 1
Added 1 x Chocolate Muffin

Enter item number (or 'done' to finish): 99
Invalid item number. Please choose from the menu.

Enter item number (or 'done' to finish): done

Order placed successfully!

                   RECEIPT
Order ID : 2
Custome

## Test 7: Out of stock and low stock warning
We try to order 300 Espressos (needs 5400 g of coffee, but only about 4900 g is left). The order is refused. After that we set the cups to 80 (the limit is 100) to trigger the low stock warning.

In [22]:
run_with_inputs(add_record, ["Test", "1", "300", "done"], menu, test_orders, test_inventory, "staff")

print("\n--- Low stock warning ---")
test_inventory["cups"]["quantity"] = 80
check_low_stock(test_inventory)
display_inventory(test_inventory)


                    MENU

--- COFFEE ---
    1. Espresso               Rs.   250.00
    2. Cappuccino             Rs.   350.00
    3. Latte                  Rs.   380.00

--- BEVERAGE ---
    4. Iced Tea               Rs.   220.00
    5. Hot Chocolate          Rs.   320.00

--- FOOD ---
    6. Chocolate Muffin       Rs.   200.00
    7. Club Sandwich          Rs.   450.00
Customer name (press Enter for 'Walk-in'): Test

Enter item number (or 'done' to finish): 1
Quantity of Espresso: 300
Sorry, not enough 'coffee' in stock for Espresso.

Enter item number (or 'done' to finish): done
No items were added. Order cancelled.

--- Low stock warning ---
** LOW STOCK WARNING: cups (80 pcs left) **

                 INVENTORY
Supply             Quantity  Unit  Status
------------------------------------------------------------
coffee                 4928  g     OK
milk                  19400  ml    OK
sugar                  2980  g     OK
cups                     80  pcs   LOW


## Test 8: Searching orders
Search by customer name (partial match, not case sensitive) and by order ID.

In [23]:
print("find_orders(name='ali') - ", [o["order_id"] for o in find_orders(test_orders, name="ali")])
print("find_orders(order_id=1) - ", [o["order_id"] for o in find_orders(test_orders, order_id=1)])
print("find_orders(name='zzz') - ", find_orders(test_orders, name="zzz"))

print("\n--- Interactive search by name ---")
run_with_inputs(search_record, ["2", "sara"], test_orders)

find_orders(name='ali') -  [2]
find_orders(order_id=1) -  [1]
find_orders(name='zzz') -  []

--- Interactive search by name ---

1. Search by Order ID
2. Search by Customer Name
Choose: 2
Enter customer name: sara

1 order(s) found:

                   RECEIPT
Order ID : 1
Customer : Sara
Date     : 2026-11-14  21:53:16
Served by: staff
------------------------------------------------------------
Item                    Qty     Price       Total
------------------------------------------------------------
Cappuccino                2    350.00      700.00
Chocolate Muffin          1    200.00      200.00
------------------------------------------------------------
Subtotal                                    900.00
Tax (10%)                                    90.00
TOTAL                                       990.00


## Test 9: Manager-only menu update
An employee is refused. The manager adds a new item that uses 30 g of coffee and 1 cup.

In [24]:
MENU_FILE = "test_menu.json"
test_menu = copy.deepcopy(menu)

print("--- Employee tries to update the menu ---")
update_menu(test_menu, test_inventory, "employee")

print("\n--- Manager adds an item ---")
answers = ["1", "Mocha", "coffee", "420", "30", "0", "0", "1", "4"]
run_with_inputs(update_menu, answers, test_menu, test_inventory, "manager")
print("\nNew item stored:", test_menu["8"])

--- Employee tries to update the menu ---
Access denied. Only the manager can update the menu.

--- Manager adds an item ---

--- UPDATE MENU ---
1. Add new item
2. Change item price
3. Remove item
4. Back
Choose: 1
Item name: Mocha
Category (Coffee / Beverage / Food): coffee
Price: 420
coffee used per item (g, 0 if none): 30
milk used per item (ml, 0 if none): 0
sugar used per item (g, 0 if none): 0
cups used per item (pcs, 0 if none): 1
'Mocha' added with item number 8.

--- UPDATE MENU ---
1. Add new item
2. Change item price
3. Remove item
4. Back
Choose: 4

New item stored: {'name': 'Mocha', 'category': 'Coffee', 'price': 420.0, 'uses': {'coffee': 30, 'cups': 1}}


## Test 10: Sales report
The report is built from the test orders. It is printed and also saved to `sales_report.txt`.

In [25]:
generate_report(test_orders)

                SALES REPORT
Today (2026-11-14)
  Orders  : 2
  Revenue : Rs. 1980.00
------------------------------------------------------------
Daily revenue:
  2026-11-14  |    2 orders  |  Rs.    1980.00
------------------------------------------------------------
All time orders  : 2
All time revenue : Rs. 1980.00
------------------------------------------------------------
Items sold (highest first):
  Cappuccino                4
  Chocolate Muffin          2
(Report also saved to sales_report.txt)


## Test 11: Data files after the tests
This shows that the orders were really written to a file, and that the file can be read back by `load_data()`.

In [26]:
reloaded = load_data(ORDERS_FILE, [])
print("Orders saved in", ORDERS_FILE, ":", len(reloaded))
print("\nFirst order as stored in the JSON file:")
with open(ORDERS_FILE) as f:
    print(f.read()[:700] + "\n...")

Orders saved in test_orders.json : 2

First order as stored in the JSON file:
[
    {
        "order_id": 1,
        "customer": "Sara",
        "date": "2026-11-14",
        "time": "21:53:16",
        "cashier": "staff",
        "items": [
            {
                "item_id": "2",
                "name": "Cappuccino",
                "price": 350.0,
                "quantity": 2
            },
            {
                "item_id": "6",
                "name": "Chocolate Muffin",
                "price": 200.0,
                "quantity": 1
            }
        ],
        "subtotal": 900.0,
        "tax": 90.0,
        "total": 990.0
    },
    {
        "order_id": 2,
        "customer": "Ali",
        "date": "2026-11-14",
        "time": "21:53:16",
      
...


## Clean up the test files
We delete the test files and put the real file names back.

In [27]:
for name in ["test_missing.json", "test_bad.json", "test_bad.json.bak",
             "test_orders.json", "test_inventory.json", "test_menu.json",
             "sales_report.txt"]:
    try:
        os.remove(name)
    except FileNotFoundError:
        pass
os.rmdir("test_folder")

MENU_FILE = "menu.json"
ORDERS_FILE = "orders.json"
INVENTORY_FILE = "inventory.json"
print("Test files removed. Real files restored:", MENU_FILE, ORDERS_FILE, INVENTORY_FILE)

Test files removed. Real files restored: menu.json orders.json inventory.json


# Run the real portal
Run the cell below and use the portal normally. Logins: `manager / manager123` and `staff / staff123`.
Type your answers in the box that appears under the cell. Choose option 8 to exit.

In [ ]:
main()

        COFFEE SHOP MANAGEMENT PORTAL


Username:  manager
Password:  manager123



Welcome, manager! (Role: manager)

   MAIN MENU   (User: manager | Role: manager)
1. View Menu
2. Place an Order
3. Reprint Bill (existing order)
4. Search Orders
5. Inventory Management
6. Update Menu (manager only)
7. Sales Report
8. Logout / Exit


Enter your choice (1-8):  2



                    MENU

--- COFFEE ---
    1. Espresso               Rs.   250.00
    2. Cappuccino             Rs.   350.00
    3. Latte                  Rs.   380.00

--- BEVERAGE ---
    4. Iced Tea               Rs.   220.00
    5. Hot Chocolate          Rs.   320.00

--- FOOD ---
    6. Chocolate Muffin       Rs.   200.00
    7. Club Sandwich          Rs.   450.00


Customer name (press Enter for 'Walk-in'):  3

Enter item number (or 'done' to finish):  4
Quantity of Iced Tea:  2


... (4 outputs left out)


--- UPDATE MENU ---
1. Add new item
2. Change item price
3. Remove item
4. Back


Choose:  1
Item name:  Cola
Category (Coffee / Beverage / Food):  Beverage
Price:  120.0
coffee used per item (g, 0 if none):  0
milk used per item (ml, 0 if none):  0
sugar used per item (g, 0 if none):  30
cups used per item (pcs, 0 if none):  1


'Cola' added with item number 8.

--- UPDATE MENU ---
1. Add new item
2. Change item price
3. Remove item
4. Back


Choose:  1
